In [1]:
from ecostyles import EcoStyles
styles = EcoStyles()
styles.register_and_enable_theme(theme_name="newsletter")

import pandas as pd, altair as alt, os

# for folder in ['data', 'raw', 'charts']:
#     os.makedirs(folder, exist_ok=True)


Data sources:
- Wimbledon prize money: https://www.wimbledon.com/en_GB/about_wimbledon/prize_money_and_finance.html
- Grand slam prize pots: https://www.tennisnerd.net/prize-money#slams

### Overall prize money

In [2]:
df = pd.read_excel('data/prize-money.xlsx')

# Clean every column (strip whitespace and remove ',) then convert to numeric
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.strip().str.replace(',', '').astype(float)

df

# # There is no row for 2020, so add one with missing values

# idx = pd.date_range(df['Year'].min(), df['Year'].max(), freq='YE')
# df = df.set_index('Year').reindex(idx, fill_value=None).reset_index()

# df

,Year,Gentlemen's Singles,Gentlemen's Doubles (pair),Ladies' Singles,Ladies' Doubles (pair),Mixed Doubles (pair),Total for meeting
0,1968,2000.0,800.0,750.0,500.0,450.0,26150.0
1,1969,3000.0,1000.0,1500.0,600.0,500.0,33370.0
2,1970,3000.0,1000.0,1500.0,600.0,500.0,41650.0
3,1971,3750.0,750.0,1800.0,450.0,375.0,37790.0
4,1972,5000.0,1000.0,3000.0,600.0,500.0,50330.0
5,1973,5000.0,1000.0,3000.0,600.0,500.0,52400.0
6,1974,10000.0,2000.0,7000.0,1200.0,1000.0,97100.0
7,1975,10000.0,2000.0,7000.0,1200.0,1000.0,114875.0
8,1976,12500.0,3000.0,10000.0,2400.0,2000.0,157740.0
9,1977,15000.0,6000.0,13500.0,5200.0,3000.0,222540.0


Graf and Edberg took home £148.5k and £165k respectively in 1988, today's winners pocket £3m.

- Womens % rise: 1,920%
- Mens % rise: 1,718%

CPI Inflation during that time. Index for July 1988 49.7, for May 2025 138.4

Adjusted for inflation:
- Womens prize money: £148.5k * 138.4 / 49.7 = £413.5291750503k -> 625.4628513447
- Mens prize money: £165k * 138.4 / 49.7 = £459.476861167 -> 553 % rise


Jan 1988 to May 2025 (138.4-48.4)/48.4
July 1988 to May 2025: (138.4-49.7)/49.7 = 1.7847082495


CPI was 49.7 in July 1988 and 138.4 in May 2025


How many tournamaents since prize money was first introduced? (1968)
- Started in 1877, less 4 editions for WW1, 6 for WW2.
- So 1877-1968 = 91 years, less 10 for WW1 and WW2 = 81 editions

In [4]:
# Add column for % change of total prize money from previous year
df['% change'] = (df['Total for meeting'].pct_change() * 100).round(1)

# Add column for decade
df['decade'] = df['Year'].astype(str).str.slice(0, 3) + '0s'

df

/var/folders/6z/lf62926s49ldk22fm5t41t8r0000gp/T/ipykernel_31934/3790235980.py:2: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['% change'] = (df['Total for meeting'].pct_change() * 100).round(1)


,Year,Gentlemen's Singles,Gentlemen's Doubles (pair),Ladies' Singles,Ladies' Doubles (pair),Mixed Doubles (pair),Total for meeting,% change,decade
0,1968,2000.0,800.0,750.0,500.0,450.0,26150.0,NaN,1960s
1,1969,3000.0,1000.0,1500.0,600.0,500.0,33370.0,27.6,1960s
2,1970,3000.0,1000.0,1500.0,600.0,500.0,41650.0,24.8,1970s
3,1971,3750.0,750.0,1800.0,450.0,375.0,37790.0,-9.3,1970s
4,1972,5000.0,1000.0,3000.0,600.0,500.0,50330.0,33.2,1970s
5,1973,5000.0,1000.0,3000.0,600.0,500.0,52400.0,4.1,1970s
6,1974,10000.0,2000.0,7000.0,1200.0,1000.0,97100.0,85.3,1970s
7,1975,10000.0,2000.0,7000.0,1200.0,1000.0,114875.0,18.3,1970s
8,1976,12500.0,3000.0,10000.0,2400.0,2000.0,157740.0,37.3,1970s
9,1977,15000.0,6000.0,13500.0,5200.0,3000.0,222540.0,41.1,1970s


Adjust annual % rises with RPI inflation.

In [5]:
rpi = pd.read_csv('data/ons-rpi.csv')

# Filter to only May of each year (as latest month we have is May)
rpi = rpi[rpi['Title'].str.contains('MAY')].reset_index(drop=True).copy()

# Filter to after 1968
rpi = rpi[rpi['Title'] >= '1968 MAY'].reset_index(drop=True).copy()

rpi.columns = ['date', 'rpi'] 
# Add year column 
rpi['Year'] = rpi['date'].str.slice(0, 4).astype(int)

# # Add column for % change of RPI from previous year
# rpi['% change'] = (rpi['Value'].pct_change() * 100).round(1)

# Merge dataframe with total prize money
total = df[['Year', 'Total for meeting']].copy()

total = total.merge(rpi, on='Year', how='left')

total['rpi'] = total['rpi'].astype(float)
 
# Rebase rpi to most recent value (2025) prices = 100
total['rpi'] = total['rpi'] / total['rpi'].iloc[-1]

# Add inflation adjusted prize money, in 2025 prices
total['total_adj'] = (total['Total for meeting'] / total['rpi']).round(0)

# Add decade column
total['decade'] = total['Year'].astype(str).str.slice(0, 3) + '0s'

# Calculate annual % percentage change
total['annual rise'] = (total['total_adj'].pct_change() * 100).round(1)

# Calculate avarage percentage change per decade. Which decades have seen the largest rises?
total.groupby('decade')['annual rise'].mean().sort_values(ascending=False)


FileNotFoundError: [Errno 2] No such file or directory: 'data/ons-rpi.csv'

In [6]:
total

NameError: name 'total' is not defined

1.09766105868

1.4078947368

 larger than hit to tournament finances and burst of inflation: 2025 will finally see the total prize pool rise above that of the 2019 pre-pandemic tournament. 

CPI May 2019: 107.9
CPI May 2025: 138.4

2019 prize is £48.74m, in 2025 price after CPI

CPIH May 2019: 107.0


In [7]:
# Calculate inflation adjusted prize money absolute difference from previous year
total['total_adj_diff'] = total['total_adj'].diff()

total


NameError: name 'total' is not defined

Chart. winning prize money by event

In [8]:
df_temp = df.melt(id_vars=['Year'], var_name='category', value_name='value')

df_temp['Year'] = df_temp['Year'].astype(str) + '-01-01'

df_temp['category'] = df_temp['category'].str.replace(' (pair)', '').str.strip()

In [9]:
df_temp['category'].unique()

array(["Gentlemen's Singles", "Gentlemen's Doubles", "Ladies' Singles",
       "Ladies' Doubles", 'Mixed Doubles', 'Total for meeting',
       '% change', 'decade'], dtype=object)

In [3]:
base = alt.Chart(df_temp).transform_filter(
    alt.datum.category != 'Total for meeting'
).encode(
    alt.X('Year:T').axis(format='%Y', labelFlush=False),
    alt.Y('value:Q').axis(
        labelExpr="indexof(datum.label, 0) == 0 ? '£0' : '£' + datum.value / 1000000 + 'm'",
        title='Winning prize money, event',
        titleFontSize=13
    ),
    alt.Color('category:N').legend(None)
)

lines = base.mark_line(strokeWidth=2.9, interpolate='step')

labels = base.mark_text(
    dy=alt.expr('{"Gentlemen\'s Singles": 6, "Ladies\' Singles": -6, "Ladies\' Doubles": -6, "Gentlemen\'s Doubles": 6}[datum.category]'),
    fontSize=14
).encode(
    alt.X('Year:T').aggregate('max'),
    alt.Y('value:Q').aggregate({'argmax': 'Year'}),
    alt.Text('category:N').aggregate({'argmax': 'Year'})
)

# Add shade area for missing 2020 tournament year
area = alt.Chart(pd.DataFrame({'x1': ['2020-01-01'], 'x2': ['2021-01-01']})).mark_rect(opacity=0.4).encode(
    alt.X('x1:T'),
    alt.X2('x2:T')
)

chart = alt.layer(area, lines, labels).properties(
    width=350,
    height=280
).configure_axis(
    labelFontSize=14
)
chart.display()

styles.save(chart, 'raw', 'fig4-prize-money', width=350, height=280)

NameError: name 'df_temp' is not defined

<br>

Total prize pot against inflation

In [158]:
df_total = df_temp[df_temp['category'] == 'Total for meeting'].copy()

bars = alt.Chart(df_total).mark_bar(strokeWidth=2.9).encode(
    alt.X('Year:T').axis(format='%Y', labelFlush=False),
    alt.Y('value:Q').axis(
        title='Prize pot, total for meeting',
        labelExpr="indexof(datum.label, 0) == 0 ? '£0' : '£' + datum.value / 1000000 + 'm'",
        titleFontSize=13,
        tickCount=11
    )
)

# Add shade area for missing 2020 tournament year
area = alt.Chart(pd.DataFrame({'x1': ['2020-01-01'], 'x2': ['2021-01-01']})).mark_bar(color='lightgrey', opacity=0.3, width=9).encode(
    alt.X('x1:T').axis(format='%Y', labelFlush=False)
)

chart = alt.layer(area, bars).properties(
    width=350,
    height=250
).configure_axis(
    labelFontSize=14
)
chart.display()

styles.save(chart, 'raw', 'fig5-prize-total', width=350, height=250)


alt.LayerChart(...)

In [ ]:
chart

In [159]:
df_total

,Year,category,value
290,1968-01-01,Total for meeting,26150.0
291,1969-01-01,Total for meeting,33370.0
292,1970-01-01,Total for meeting,41650.0
293,1971-01-01,Total for meeting,37790.0
294,1972-01-01,Total for meeting,50330.0
295,1973-01-01,Total for meeting,52400.0
296,1974-01-01,Total for meeting,97100.0
297,1975-01-01,Total for meeting,114875.0
298,1976-01-01,Total for meeting,157740.0
299,1977-01-01,Total for meeting,222540.0


---

### Distribution of prize money

Prize money by round:
- 2010: https://web.archive.org/web/20100704205621/http://aeltc2010.wimbledon.org/en_GB/about/pdf/Prize_Money_2010.pdf
- 2011: https://web.archive.org/web/20110626042544/http://aeltc2011.wimbledon.com/content/documents/visiting-wimbledon/2011-prizemoney.pdf


In [4]:
df = pd.read_csv('data/prize_money_2009_2010_2025_2026.csv')

# Remove if not in 2010 or 2025
df = df[df['Year'].isin([2010, 2026])].copy()

# Replace '‑' with '-'
df['Round'] = df['Round'].str.replace('‑', '-')

# # Add ordering column for round
# order = ['First Round Losers', 'Second Round Losers', 'Third Round Losers', 'Fourth Round Losers', 'Quarter-Finalists', 'Semi-Finalists', 'Runner-up', 'Winner']
# # Add column with 1-8 for round
# #
# df['round'] = pd.Categorical(df['Round'].str.strip(), categories=order, ordered=True)
# df['round_num'] = df['round'].cat.codes + 1
df


,Year,Round,Prize_Per_Player_GBP,Total_Prize_GBP,Increase_Pct
0,2010,Winner,1000000,1000000,17.6
1,2010,Runner-up,500000,500000,17.6
2,2010,Semi-Finalists,250000,500000,17.6
3,2010,Quarter-Finalists,125000,500000,17.6
4,2010,Fourth Round Losers,62500,500000,17.4
5,2010,Third Round Losers,31250,500000,6.8
6,2010,Second Round Losers,18750,600000,5.6
7,2010,First Round Losers,11250,720000,4.7
24,2026,Winner,3600000,3600000,20.0
25,2026,Runner-up,1800000,1800000,18.0


Recent years has seen some narrowing of earnings inequality. The three-times growth in earnings for singles’ champions since 2010 (£1m to 3m) is half the growth seen by those reaching the first round (£11k to £66k). Yet this hasn't been enough to help the affordability issues felt by many players across the circuit, indeed the players have  

, {add better word here} relatively low prize money across the wider circuit, especially for early rounds, 


In [324]:
print(f"{(11250 / df[df['Year']==2010]['Total_Prize_GBP'].sum()):.2%}")

0.23%


**Pyramid chart**

In [303]:
3e6 % 1e6

0.0

In [5]:
# 1. share-of-total within each year
df['share'] = df.groupby('Year')['Prize_Per_Player_GBP'].transform(lambda x: x / x.sum())

# 2. flip sign for 2010 so bars diverge from centre
df['prize_plot'] = df.apply(lambda r: -r['Prize_Per_Player_GBP'] if r['Year']==2010 else r['Prize_Per_Player_GBP'], axis=1)

order = ['First Round Losers','Second Round Losers','Third Round Losers','Fourth Round Losers',
         'Quarter-Finalists','Semi-Finalists','Runner-up','Winner']

# Add formatted prize column, e.g. 99000 to £99k, 1520000 to £1.52m, 3000000 to £3m
df['prize_label'] = df['Prize_Per_Player_GBP'].apply(lambda x: f'£{x/1000:.0f}k' if (x < 1000000) else f'£{x/1000000:.2f}m' if (x > 1e6 and x%1e6!=0) else f'£{x/1e6:.0f}m')

chart = alt.Chart(df).mark_bar().encode(
    alt.Y('Round:N').sort(order).title('').axis(
        titleAngle=0,
        titleAnchor='start',
        titleX=2,
        titleY=-2,
        titleFontSize=13
    ),
    alt.X('prize_plot:Q').title('Prize winnings by round').axis(
        titleFontSize=13,
        gridOpacity=0.2,
        # labelExpr="datum.value < 0 ? '£' + format(-datum.value, ) : datum.value == 0 ? '0' : '£' + datum.value",
        labelExpr="abs(datum.value) >= 1000000 ? '£' + abs(datum.value / 1000000) + 'm' : datum.value == 0 ? '0' : '£' + abs(datum.value / 1000) + 'k'"
    ).scale(
     domain=[-1.9e6, 3e6],
        nice=False
    ),
    alt.Color('Year:N').scale().legend(orient='top-right', labelFontSize=14, symbolSize=115),
    tooltip=['Year','Round','share:Q']
).properties(width=350)


text = alt.Chart(df).mark_text(
    align=alt.expr('datum.prize_plot < -1500000 ? "left" : datum.prize_plot < 0 ? "right" : datum.prize_plot < 1500000 ? "left" : "right"'),
    color=alt.expr('datum.prize_plot < -1500000 ? "#fff" : datum.prize_plot < 0 ? "grey" : datum.prize_plot < 1500000 ? "grey" : "#fff"'),
    dx=alt.expr('datum.prize_plot < -1500000 ? 4 : datum.prize_plot < 0 ? -4 : datum.prize_plot < 1500000 ? 4 : -4'),
    fontSize=13
).encode(
    alt.Y('Round:N').sort(order),
    alt.X('prize_plot:Q'),
    alt.Text('prize_label:N')
)

# Draw a thin vertical centre line
centre = alt.Chart(pd.DataFrame({'x':[0]})).mark_rule(color='gray').encode(x='x:Q')

chart = (centre + chart + text).configure_axis(grid=False, labelFontSize=14)

# # Create pyramid chart with 2010 distribution on left and 2025 on right
# chart = alt.Chart(df).transform_filter(
#     alt.datum.Year == 2010
# ).mark_bar(strokeWidth=2.9).encode(
#     alt.Y('round:O').sort(order).axis(None),
#     alt.X('prize_plot:Q').axis(
#         title='Prize money, round',
#         titleFontSize=13,
#         tickCount=11
#     ).scale(reverse=True),
#     alt.Color('Year:N').scale().legend(orient='top-right', labelFontSize=14, symbolSize=115),
# )

# right = alt.Chart(df).transform_filter(
#     alt.datum.Year == 2025
# ).mark_bar(strokeWidth=2.9).encode(
#     alt.Y('round:O').sort(alt.EncodingSortField('round_num', order='ascending')).axis(None),
#     alt.X('Prize_Per_Player_GBP:Q').axis(
#         title='Prize money, round',
#         titleFontSize=13,
#         tickCount=11
#     )
# )

chart.display()
styles.save(chart, 'raw', 'fig6a-prize-distribution', width=350, height=270)

alt.LayerChart(...)

**Divering 100%-bar butterfly**

In [6]:
import pandas as pd, altair as alt

df = pd.read_csv('data/prize_money_2009_2010_2025_2026.csv')
df = df[df['Year'].isin([2010, 2026])].copy()
df['Round'] = df['Round'].str.replace('‑', '-')

In [7]:
# 1. share-of-total within each year
df['share'] = df.groupby('Year')['Prize_Per_Player_GBP'].transform(lambda x: x / x.sum())

# 2. flip sign for 2010 so bars diverge from centre
df['share_plot'] = df.apply(lambda r: -r['share'] if r['Year']==2010 else r['share'], axis=1)

order = ['First Round Losers','Second Round Losers','Third Round Losers','Fourth Round Losers',
         'Quarter-Finalists','Semi-Finalists','Runner-up','Winner']

chart = (alt.Chart(df)
         .mark_bar()
         .encode(
             alt.Y('Round:N').sort(order).title('').axis(
                 titleAngle=0,
                 titleAnchor='start',
                 titleX=2,
                 titleY=-2,
                 titleFontSize=13
             ),
             alt.X('share_plot:Q').title('Share of singles prize pool').axis(
                 titleFontSize=13,
                 gridOpacity=0.2,
                 labelExpr="datum.value < 0 ? -datum.value * 100 + '%' : datum.value == 0 ? '0' : datum.value * 100 + '%'",
             ).scale(
                 domain=[-0.55, 0.55],
                 nice=False
             ),
             alt.Color('Year:N').scale().legend(orient='top-right', labelFontSize=14, symbolSize=115),
             tooltip=['Year','Round','share:Q']
         )
         .properties(width=350)
)

text = alt.Chart(df).mark_text(
    align=alt.expr('datum.share_plot < -0.2 ? "left" : datum.share_plot < 0 ? "right" : datum.share_plot < 0.2 ? "left" : "right"'),
    color=alt.expr('datum.share_plot < -0.2 ? "#fff" : datum.share_plot < 0 ? "grey" : datum.share_plot < 0.2 ? "grey" : "#fff"'),
    dx=alt.expr('datum.share_plot < -0.2 ? 4 : datum.share_plot < 0 ? -4 : datum.share_plot < 0.2 ? 4 : -4'),
    fontSize=13
).encode(
    alt.Y('Round:N').sort(order),
    alt.X('share_plot:Q'),
    alt.Text('share:Q').format('.1~%')
)

# Draw a thin vertical centre line
centre = alt.Chart(pd.DataFrame({'x':[0]})).mark_rule(color='gray').encode(x='x:Q')

chart = (centre + chart + text).configure_axis(grid=False, labelFontSize=14)

chart.display()

styles.save(chart, 'raw', 'fig6-prize-distribution', width=350, height=270)


alt.LayerChart(...)

In [8]:
df.groupby('Year')['Total_Prize_GBP'].sum()

Year
2010     4820000
2026    23632000
Name: Total_Prize_GBP, dtype: int64

In [9]:
# Calculate 1st round prize over total prize pool for singles tournament
print(f"2010 1st round: {(11250 / 4820000):.2%}")
print(f"2025 1st round: {(66000 / 19414000):.2%}")

print(f"2010 winner: {(1e6 / 4820000):.2%}")
print(f"2025 winner: {(3e6 / 19414000):.2%}")


2010 1st round: 0.23%
2025 1st round: 0.34%
2010 winner: 20.75%
2025 winner: 15.45%


In [357]:
df.groupby('Year')['Prize_Per_Player_GBP'].transform(lambda x: x / x.sum())


0     0.500313
1     0.250156
2     0.125078
3     0.062539
4     0.031270
5     0.015635
6     0.009381
7     0.005629
16    0.479846
17    0.243122
18    0.123960
19    0.063980
20    0.038388
21    0.024312
22    0.015835
23    0.010557
Name: Prize_Per_Player_GBP, dtype: float64

In [356]:
# Calculate Prize_Per_Player_GBP as share of Total_Prize_GBP for each year
df['share'] = df.groupby('Year')['Prize_Per_Player_GBP'].transform(lambda x: x / x.sum())



Year
2010     4820000
2025    19414000
Name: Total_Prize_GBP, dtype: int64

In [268]:
df_wide = df.pivot(index='Round', columns='Year', values='Prize_Per_Player_GBP').reset_index()
df_wide['increase'] = df_wide[2025] / df_wide[2010]
df_wide['increase'] = ((df_wide['increase'] - 1) * 100).round(0).astype(int).astype(str) + '%'
# Sort according to 'order'
order = ['First Round Losers','Second Round Losers','Third Round Losers','Fourth Round Losers',
         'Quarter-Finalists','Semi-Finalists','Runner-up','Winner']
df_wide = df_wide.sort_values('Round', key=lambda x: x.map(lambda y: order.index(y)))
df_wide


Year,Round,2010,2025,increase
0,First Round Losers,11250,66000,487%
4,Second Round Losers,18750,99000,428%
6,Third Round Losers,31250,152000,386%
1,Fourth Round Losers,62500,240000,284%
2,Quarter-Finalists,125000,400000,220%
5,Semi-Finalists,250000,775000,210%
3,Runner-up,500000,1520000,204%
7,Winner,1000000,3000000,200%


In [254]:
import numpy as np
def gini(x):  # simple Gini for discrete rounds
    x = np.sort(x); n = len(x); cum = np.cumsum(x)
    return (n+1 - 2*(cum / cum[-1]).sum()) / n
for y, grp in df.groupby('Year'):
    print(y, gini(grp['Prize_Per_Player_GBP'].values))

2010 0.6282833020637899
2025 0.5932501599488164


Equivalent gini to Namibia (59.1%).

In [ ]:
alt.Chart(df)

In [258]:
# 1. share-of-total within each year
df['share'] = df.groupby('Year')['Prize_Per_Player_GBP'].transform(lambda x: x / x.sum())

# 2. flip sign for 2010 so bars diverge from centre
df['share_plot'] = df.apply(lambda r: -r['share'] if r['Year']==2010 else r['share'], axis=1)

order = ['First Round Losers','Second Round Losers','Third Round Losers','Fourth Round Losers',
         'Quarter-Finalists','Semi-Finalists','Runner-up','Winner']

chart = (alt.Chart(df)
         .mark_bar()
         .encode(
             alt.Y('Round:N').sort(order).title('').axis(
                 titleAngle=0,
                 titleAnchor='start',
                 titleX=2,
                 titleY=-2,
                 titleFontSize=13
             ),
             alt.X('Prize_Per_Player_GBP:Q').title('Share of singles prize pool').axis(
             ).scale(
                 nice=False
             ),
             alt.Color('Year:N').scale().legend(orient='top-right', labelFontSize=14, symbolSize=115),
             tooltip=['Year','Round','share:Q']
         )
         .properties(width=350)
)


# Draw a thin vertical centre line
centre = alt.Chart(pd.DataFrame({'x':[0]})).mark_rule(color='gray').encode(x='x:Q')

chart = (centre + chart).configure_axis(grid=False, labelFontSize=14)

chart.display()

# styles.save(chart, 'raw', 'fig6-prize-distribution', width=350, height=270)


alt.LayerChart(...)